In [ ]:
import sys
from pathlib import Path

CWD = Path(__name__).resolve().parent
sys.path.append(CWD)

DATASET_FILE = CWD / "result.json"
OUT_CHECKPOINT_FILE = CWD / "leanrag_checkpoint.json"

secrets = CWD / "secrets.env"

if not secrets.is_file():
    raise ValueError(f"secrets file at '{secrets}' does not exist")

from dotenv import load_dotenv
load_dotenv(secrets)

### 1. Build leaf descriptions with embeddings

In [ ]:
from typing import TypedDict
import asyncio
import json, ijson
import dspy
import litellm
import yake
import utils.mg_driver as mg_driver
from utils import get_chunk_from_id

LEAF_PARSE_BATCH_SIZE = 10
DESC_GEN_MODEL = "bedrock/us.amazon.nova-pro-v1:0"
EMBED_MODEL = "bedrock/amazon.titan-embed-text-v2:0"

class LeafEntityData(TypedDict):
    description: str
    embedding: list[float]    

G0_entity_data : dict[str, LeafEntityData] = { } # holds leaf entity descriptions + embeddings, indexed by entity key

# YAKE
kw_extractor = yake.KeywordExtractor(
    lan = "en",
    n=3,
    dedupLim=0.9,
    top=10,
    features=None
)
class _GenerateDescription(dspy.Signature):
    """Using only the provided context sentences and key phrases related to the entity,
      create a 2-3 sentence description for the entity. Do not invent facts."""

    entity_name = dspy.InputField(desc="The name of the entity.")
    key_phrases = dspy.InputField(desc="Relevant key phrases to the entity.")
    context_sentences = dspy.InputField(desc="The context from which to generate a description.")
    entity_description = dspy.OutputField(desc="A 2-3 sentence description of the entity from the context sentences.")

class NodeDescriptionGenerator:
    """
    Generate descriptions for a node
    """

    def __init__(self, model: str) -> None:
        self._lm = dspy.LM(model)

    async def generate(self, entity_name: str, key_phrases:list[str], context_sentences:list[str]) -> str|None:
        try:
            with dspy.context(lm=self._lm):
                pred = await dspy.Predict(_GenerateDescription).acall(entity_name=entity_name, key_phrases=key_phrases, context_sentences=context_sentences)
            out_desc = pred['entity_description']
            if not out_desc:
                raise Exception("generated empty node description")
        except Exception as e:
            print(f"Node description generation error: {type(e).__name__}: {e}")
            return None
        
        return out_desc
        
description_generator = NodeDescriptionGenerator(model=DESC_GEN_MODEL)

# parse full graph in batches
async def gen_leaf_descriptions_embeddings(dataset):
    entity_count:list[dict] = await mg_driver.read("MATCH (n:Entity) RETURN count(n) AS c")
    if not 'c' in entity_count[0].keys():
        raise ValueError("get entity count cypher failed")
    entity_count= entity_count[0]['c']
    print(f"entities to parse: {entity_count}")
    print("progress:")

    for i in range(entity_count//LEAF_PARSE_BATCH_SIZE):
        entity_batch = await mg_driver.read("""
            MATCH (node:Entity) RETURN node ORDER BY node.key SKIP $offset LIMIT $batch_size
            """,
            {
                "offset": i*LEAF_PARSE_BATCH_SIZE,
                "batch_size": LEAF_PARSE_BATCH_SIZE
            })

        for _e in entity_batch:
            entity = _e['node']
            
            #TODO: backing checkpoint json; use as persitent cache. if checkpoint contains values, then load them into memory instead of recomputing
            #TODO: catch errors

            # gather incident edges:
            incident = (await mg_driver.read(\
                """
                MATCH (:Entity {key:$key})-[r]->()
                RETURN r.source_chunk_ids AS src_chunks
                """, {"key": entity['key']}))
            
            if not incident:
                continue

            incident_chunk_ids:set = set()
            for ie in incident:
                if ie.keys and 'src_chunks' in ie.keys() and ie['src_chunks']:
                    for src in ie['src_chunks']:
                        incident_chunk_ids.add(src)
            
            incident_chunk_texts = [get_chunk_from_id(dataset, cid)['raw_text'] for cid in incident_chunk_ids]
            
            # get keywords from incident chunk texts
            key_phrases = kw_extractor.extract_keywords(" ".join(incident_chunk_texts))

            # generate description
            node_desc = await description_generator.generate(\
                entity_name=entity['name'],key_phrases=key_phrases,context_sentences=incident_chunk_texts)
            
            # embed description
            embed_desc = (await litellm.aembedding(model=EMBED_MODEL, input=node_desc))['data'][0].embedding

            # update leaf entity data entry for the current entity
            if entity['key'] in G0_entity_data.keys():
                print("detected a non-unique leaf entity! (skipping)")
                continue

            G0_entity_data[entity['key']] = {'description': node_desc, 'embedding': embed_desc}
            incr_progress()

        #TODO: execute generation + update in batch concurrently

_prog = 0
def incr_progress():
    global _prog
    _prog+=1
    print(f"{".." if _prog%20!=0 else "\n.."}{_prog}", end="")

dataset = {}
with open(DATASET_FILE, "r") as data_file:
    dataset = json.load(data_file)

await mg_driver.init()
await gen_leaf_descriptions_embeddings(dataset)
await mg_driver.close()

print(G0_entity_data)

In [ ]:
# (checkpoint:) save G0_entity_data
import json 

with open(OUT_CHECKPOINT_FILE, "w") as out_file:
    out_file.write("{\n")
    out_file.write('"G0_entity_data": ')
    json.dump(G0_entity_data, out_file, ensure_ascii=False)
    out_file.write("\n}")